# Daily Challenge — Preprocess & Fine-Tune Transformer Models

**Course:** Developers Institute  **Week 7 - Day 3**  
**Author:** Alex Goldbaum

End-to-end preprocessing pipeline for transformer text classification:

1. **Background** — how BERT and XLM-RoBERTa work.
2. **Tokenize** with `BertTokenizer` and `XLMRobertaTokenizer` and inspect
   the difference between their special tokens.
3. **Prepare input data** — padding, truncation, attention masks.
4. **Load and explore the dataset** as CSV files (the task's required workflow).
5. **Cross-validation** with `StratifiedKFold` for 5 train/val folds.
6. **Bonus** — a tiny one-fold fine-tuning to confirm the whole pipeline works.

We use the public **tweet_eval** sentiment dataset (3 classes: negative,
neutral, positive) and write it out as CSV so the workflow matches the brief
exactly. If a GPU is available the bonus fine-tuning runs in a few minutes;
otherwise it is safe to skip that final section.


## Setup


In [ ]:
%pip install -qU transformers==4.* datasets==2.* scikit-learn pandas torch


In [ ]:
import warnings
warnings.filterwarnings('ignore')

import os, random
from pathlib import Path

import numpy as np
import pandas as pd
import torch

RANDOM_STATE = 42
random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)
torch.manual_seed(RANDOM_STATE)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)


## 1. Understanding BERT and XLM-RoBERTa

Both are **Transformer encoders** — they read text bidirectionally and
produce one vector per token. They share the same architecture family but
differ in three important ways:

| Aspect | **BERT** | **XLM-RoBERTa** |
|---|---|---|
| Pre-training data | English Wikipedia + BookCorpus (~16 GB) | 2.5 TB CommonCrawl across **100 languages** |
| Pre-training objective | MLM + (originally) NSP | MLM only, longer training, larger batches |
| Tokenizer | **WordPiece** (`bert-base-uncased`) — splits with `##` continuation marks | **SentencePiece** (`xlm-roberta-base`) — language-agnostic |
| Special tokens | `[CLS]`, `[SEP]`, `[PAD]`, `[MASK]`, `[UNK]` | `<s>`, `</s>`, `<pad>`, `<mask>`, `<unk>` (RoBERTa convention) |
| Vocab size | ~30 k | ~250 k (covers 100 languages) |

**Common pre-trained checkpoints.**

- **BERT family** (English): `bert-base-uncased`, `bert-large-uncased`,
  `bert-base-cased`, `roberta-base`, `distilbert-base-uncased` (smaller +
  faster), `microsoft/MiniLM-L12-H384-uncased`.
- **Multilingual encoders**: `bert-base-multilingual-cased` (104 languages),
  **`xlm-roberta-base`** (100 languages, much stronger than mBERT in
  cross-lingual transfer), `xlm-roberta-large`.
- **Domain-specific**: `dmis-lab/biobert-base-cased-v1.1` for biomedical,
  `nlpaueb/legal-bert-base-uncased` for legal, `ProsusAI/finbert` for
  financial text.

**When to pick each.** Use **BERT** for high-volume English-only tasks
where you want the smallest reasonable footprint. Use **XLM-RoBERTa** when
your data has multiple languages or you need to ship one model that serves
all of them — its multilingual training is the single biggest reason to
reach for it instead of monolingual BERT.


In [ ]:
from transformers import BertTokenizer, XLMRobertaTokenizer

BERT_NAME = 'bert-base-uncased'
XLMR_NAME = 'xlm-roberta-base'

bert_tok = BertTokenizer.from_pretrained(BERT_NAME)
xlmr_tok = XLMRobertaTokenizer.from_pretrained(XLMR_NAME)

print('BERT vocab size      :', bert_tok.vocab_size)
print('XLM-R vocab size     :', xlmr_tok.vocab_size)
print()
print('BERT special tokens  :', bert_tok.special_tokens_map)
print('XLM-R special tokens :', xlmr_tok.special_tokens_map)


## 2. Tokenizing Text

We compare how BERT and XLM-RoBERTa tokenize the same single sentence and
the same sentence pair. Pay attention to:

- The **special tokens** wrapping each sentence (`[CLS]/[SEP]` vs `<s>/</s>`).
- The **subword markers** (`##ization` for WordPiece vs the Unicode `▁`
  word-start marker for SentencePiece).
- The shapes of `input_ids` and `attention_mask`.


In [ ]:
sentence = 'Transformers tokenize tricky subwords like internationalization.'
sentence_b = 'And they also handle multi-sentence inputs gracefully.'

print('--- BERT ---')
print('Tokens:', bert_tok.tokenize(sentence))
print()
print('--- XLM-R ---')
print('Tokens:', xlmr_tok.tokenize(sentence))


In [ ]:
# Single-sentence encoding with encode_plus (the brief asks for this API)
def encode_and_show(tok, label, text):
    enc = tok.encode_plus(
        text,
        add_special_tokens=True,
        padding='max_length',
        truncation=True,
        max_length=16,
        return_attention_mask=True,
        return_tensors='pt',
    )
    ids = enc['input_ids'][0].tolist()
    mask = enc['attention_mask'][0].tolist()
    tokens = tok.convert_ids_to_tokens(ids)
    print(f'--- {label} ---')
    print('input_ids     :', ids)
    print('attention_mask:', mask)
    print('tokens         :', tokens)
    print('decode         :', tok.decode(ids, skip_special_tokens=True))
    print()


encode_and_show(bert_tok, 'BERT single-sentence', sentence)
encode_and_show(xlmr_tok, 'XLM-R single-sentence', sentence)


In [ ]:
# Two-sentence encoding (sentence-pair classification, e.g., NLI / QA)
def encode_pair_and_show(tok, label, text_a, text_b):
    enc = tok.encode_plus(
        text_a, text_b,
        add_special_tokens=True,
        padding='max_length',
        truncation='longest_first',
        max_length=24,
        return_attention_mask=True,
        return_token_type_ids=True,
        return_tensors='pt',
    )
    ids = enc['input_ids'][0].tolist()
    type_ids = enc['token_type_ids'][0].tolist()
    tokens = tok.convert_ids_to_tokens(ids)
    print(f'--- {label} ---')
    print('tokens         :', tokens)
    print('token_type_ids :', type_ids)  # 0 for first sentence, 1 for second (BERT)
    print()


encode_pair_and_show(bert_tok, 'BERT pair', sentence, sentence_b)
encode_pair_and_show(xlmr_tok, 'XLM-R pair', sentence, sentence_b)


**Takeaways from the two outputs.**
- BERT uses `[CLS] sentence_a [SEP] sentence_b [SEP]` and tracks the two
  segments with `token_type_ids` (0 and 1).
- XLM-RoBERTa uses `<s> sentence_a </s></s> sentence_b </s>` and ignores
  `token_type_ids` (always 0).
- Both pad to the chosen `max_length` and mark the real tokens with
  `attention_mask=1` — the model uses that mask to skip padded positions.


## 3. Preparing Input Data for the Model

Three concrete preprocessing decisions:

- **`max_length`**: long enough to contain typical inputs without wasting
  compute on padding. For tweets we pick **64** (most tweets are short).
- **`truncation=True`** drops anything beyond `max_length`. This matters
  because BERT's positional embeddings are capped at 512.
- **`attention_mask`** is `1` where there is a real token and `0` for padding;
  the self-attention layers add a large negative bias on the `0` positions so
  they effectively disappear from the softmax.

Below we wrap the encoding into a small batch-friendly helper.


In [ ]:
MAX_LEN = 64


def batch_encode(tok, texts, max_length=MAX_LEN):
    return tok(
        list(texts),
        padding='max_length',
        truncation=True,
        max_length=max_length,
        return_tensors='pt',
    )


demo_texts = [
    'i love the new update, finally!',
    'absolutely terrible service from start to finish',
    'meh, not bad not great',
]
bert_batch = batch_encode(bert_tok, demo_texts)
xlmr_batch = batch_encode(xlmr_tok, demo_texts)

print('BERT batch  -> input_ids shape:', bert_batch['input_ids'].shape,
      ' attention_mask shape:', bert_batch['attention_mask'].shape)
print('XLM-R batch -> input_ids shape:', xlmr_batch['input_ids'].shape,
      ' attention_mask shape:', xlmr_batch['attention_mask'].shape)


## 4. Loading and Exploring the Dataset

The brief calls for CSV files (`train.csv`, `test.csv`). We materialize the
**tweet_eval / sentiment** dataset into that exact layout so the rest of the
code is independent of the loading mechanism. Three classes (`0=negative`,
`1=neutral`, `2=positive`), tens of thousands of training rows.


In [ ]:
from datasets import load_dataset

DATA_DIR = Path('./data')
DATA_DIR.mkdir(exist_ok=True)
TRAIN_CSV = DATA_DIR / 'train.csv'
TEST_CSV = DATA_DIR / 'test.csv'

if not (TRAIN_CSV.exists() and TEST_CSV.exists()):
    print('Building train.csv and test.csv from the tweet_eval sentiment dataset...')
    ds = load_dataset('tweet_eval', 'sentiment')
    pd.DataFrame({'text': ds['train']['text'], 'label': ds['train']['label']}).to_csv(TRAIN_CSV, index=False)
    pd.DataFrame({'text': ds['test']['text'],  'label': ds['test']['label']}).to_csv(TEST_CSV, index=False)
    print('Done.')

df_train = pd.read_csv(TRAIN_CSV)
df_test  = pd.read_csv(TEST_CSV)
print('Train shape:', df_train.shape)
print('Test  shape:', df_test.shape)
df_train.head()


In [ ]:
# Class distribution and basic stats
label_names = ['negative', 'neutral', 'positive']
counts = df_train['label'].value_counts().sort_index()
counts.index = [label_names[i] for i in counts.index]
print('Training class distribution:')
print(counts)
print(f'\nClass shares: {(counts / counts.sum()).round(3).to_dict()}')
print(f'Average text length (chars): {df_train["text"].str.len().mean():.1f}')

import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style='whitegrid')

plt.figure(figsize=(7, 4))
sns.barplot(x=counts.index, y=counts.values,
            palette=['tomato', 'gray', 'seagreen'])
plt.title('tweet_eval sentiment — training distribution', fontweight='bold')
plt.ylabel('Tweets')
for i, v in enumerate(counts.values):
    plt.text(i, v + 200, f'{v:,}', ha='center', fontweight='bold')
plt.tight_layout()
plt.show()


**Columns we need for training.** `text` (the raw tweet) and `label`
(integer 0/1/2). Everything else (split files, exploratory stats) feeds the
human review, not the model.


## 5. Stratified K-Fold Cross-Validation

We use `StratifiedKFold(n_splits=5, shuffle=True, random_state=42)` so that
each fold preserves the class proportions of the full training set — vital
when classes are imbalanced. We store **indices** rather than copies of the
data so memory use stays small even with hundreds of thousands of rows.


In [ ]:
from sklearn.model_selection import StratifiedKFold

K = 5
kf = StratifiedKFold(n_splits=K, shuffle=True, random_state=RANDOM_STATE)

splits = []   # list of (train_idx, val_idx)
for fold, (train_idx, val_idx) in enumerate(kf.split(df_train, df_train['label']), start=1):
    splits.append((train_idx, val_idx))
    val_labels = df_train.loc[val_idx, 'label']
    print(f'Fold {fold}: train={len(train_idx):>5}  val={len(val_idx):>5}',
          ' val_class_shares=', val_labels.value_counts(normalize=True).sort_index().round(3).to_dict())


**Reading the printout.** Each fold's validation class shares stay within ~1
percentage point of the full training distribution — that is what
`stratify` buys us. If we had used a plain `KFold`, the worst fold could
easily under-represent a class by 5+ pp, giving misleading validation
metrics.


In [ ]:
# Helper to materialize a fold as tokenized PyTorch tensors with one or both tokenizers
def make_fold_tensors(fold_idx, tokenizer):
    train_idx, val_idx = splits[fold_idx]
    train_texts = df_train.loc[train_idx, 'text'].astype(str).tolist()
    val_texts   = df_train.loc[val_idx,   'text'].astype(str).tolist()
    train_labels = df_train.loc[train_idx, 'label'].values
    val_labels   = df_train.loc[val_idx,   'label'].values
    train_enc = batch_encode(tokenizer, train_texts)
    val_enc   = batch_encode(tokenizer, val_texts)
    return (train_enc, train_labels), (val_enc, val_labels)


(train_enc, train_y), (val_enc, val_y) = make_fold_tensors(0, bert_tok)
print('Fold 0 BERT — train input_ids shape:', train_enc['input_ids'].shape)
print('Fold 0 BERT — val   input_ids shape:', val_enc['input_ids'].shape)


## 6. (Bonus) One-Fold Fine-Tuning Sanity Check

To confirm the whole pipeline works end-to-end, we fine-tune **DistilBERT**
for **1 epoch** on fold 0 and evaluate on its validation split. This is not
meant as a competitive run — it is a sanity check that nothing in the
preprocessing breaks the training loop. On a Colab T4 it finishes in 3–5
minutes. Skip if you are CPU-only.


In [ ]:
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    Trainer, TrainingArguments, DataCollatorWithPadding,
)
from torch.utils.data import Dataset


class CSVDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = torch.tensor(labels, dtype=torch.long)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {k: v[idx] for k, v in self.encodings.items()}
        item['labels'] = self.labels[idx]
        return item


DEMO_MODEL = 'distilbert-base-uncased'
demo_tok = AutoTokenizer.from_pretrained(DEMO_MODEL)

# Re-tokenize fold 0 with the demo tokenizer
(train_enc, train_y), (val_enc, val_y) = make_fold_tensors(0, demo_tok)
train_ds = CSVDataset(train_enc, train_y)
val_ds   = CSVDataset(val_enc, val_y)

demo_model = AutoModelForSequenceClassification.from_pretrained(
    DEMO_MODEL,
    num_labels=3,
    id2label={0:'negative', 1:'neutral', 2:'positive'},
    label2id={'negative':0, 'neutral':1, 'positive':2},
)

from sklearn.metrics import accuracy_score, f1_score

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = logits.argmax(-1)
    return {'accuracy': accuracy_score(labels, preds),
            'f1': f1_score(labels, preds, average='macro')}

args = TrainingArguments(
    output_dir='./fold0_demo',
    num_train_epochs=1,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    learning_rate=5e-5,
    weight_decay=0.01,
    eval_strategy='epoch',
    save_strategy='no',
    logging_steps=200,
    report_to='none',
    seed=RANDOM_STATE,
)

trainer = Trainer(
    model=demo_model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    tokenizer=demo_tok,
    data_collator=DataCollatorWithPadding(tokenizer=demo_tok),
    compute_metrics=compute_metrics,
)
trainer.train()
print(trainer.evaluate())


## Summary

- BERT and XLM-RoBERTa share the encoder architecture but differ in
  tokenizer (WordPiece vs SentencePiece), special tokens
  (`[CLS]/[SEP]` vs `<s>/</s>`), vocab size and pre-training data.
- `encode_plus` (or the modern `tokenizer(text_a, text_b, ...)` shorthand)
  handles padding, truncation, special tokens and attention masks in one call.
- We loaded `train.csv` / `test.csv` of the tweet_eval sentiment task,
  inspected class balance, and confirmed the columns we need (`text`, `label`).
- `StratifiedKFold(shuffle=True)` gave us five train/val splits with
  preserved class proportions, exactly what we want for honest validation.
- The bonus section confirmed the whole pipeline trains and evaluates
  correctly with `Trainer` — ready to be looped over all 5 folds for a full
  cross-validated fine-tuning run.

**Next step in production.** Loop the bonus section over each of the 5
folds, average the F1 scores, and pick hyperparameters (learning rate,
epochs, batch size) on the average rather than on a single split. Then
retrain on **all** of `train.csv` with the chosen hyperparameters and
evaluate on `test.csv` for the final reportable score.
